# Medicare Part D — Data Exploration

**Prerequisite:** Run `00_build_database.ipynb` first.

This notebook documents the initial data exploration process: understanding the structure of the CMS Part D geography dataset, identifying which drug categories are represented, and establishing the criteria for selecting drugs suitable for geographic analysis.

The Streamlit app (`app.py`) is the interactive version of this analysis. This notebook serves as the analytical foundation and methodology record.

In [ ]:
import sqlite3
import pandas as pd

db_path = r"C:\Users\palla\OneDrive\Documents\Coding Projects\CMS\database\cms.db"
conn = sqlite3.connect(db_path)

## Step 1: Dataset Overview

The dataset contains two geographic levels: National (one row per drug) and State (one row per drug per state). National rows are used for total cost and claim summaries. State rows are used for geographic mapping.

CMS suppresses state-level values when fewer than 11 beneficiaries received a drug in that state — those cells appear as NULL in the database.

In [ ]:
geo_summary = pd.read_sql_query("""
    SELECT Prscrbr_Geo_Lvl, COUNT(*) AS rows, COUNT(DISTINCT Gnrc_Name) AS unique_drugs
    FROM part_d_geo
    GROUP BY Prscrbr_Geo_Lvl
""", conn)

geo_summary

## Step 2: Top Drugs by Total Cost (National)

Identifying the highest-cost drugs reveals where Medicare spending is concentrated. These are the drugs most relevant for cost surveillance and access analysis.

In [ ]:
top_cost = pd.read_sql_query("""
    SELECT Gnrc_Name, Brnd_Name,
           Tot_Clms, Tot_Benes,
           ROUND(Tot_Drug_Cst, 0) AS Tot_Drug_Cst,
           ROUND(Tot_Drug_Cst / Tot_Benes, 0) AS Cost_Per_Bene
    FROM part_d_geo
    WHERE Prscrbr_Geo_Lvl = 'National'
      AND Tot_Benes IS NOT NULL
    ORDER BY Tot_Drug_Cst DESC
    LIMIT 20
""", conn)

top_cost

## Step 3: Drug Suitability for Geographic Mapping

For a state-level choropleth map to be meaningful, a drug needs data in most or all 50 states. Drugs with limited geographic reach are excluded from the app.

Two factors determine state coverage:
1. Prescribing volume — rare drugs fall below CMS reporting thresholds in smaller states
2. Route of administration — IV drugs billed under Medicare Part B will not appear in Part D data regardless of volume

In [ ]:
state_coverage = pd.read_sql_query("""
    SELECT Gnrc_Name,
           COUNT(DISTINCT Prscrbr_Geo_Desc) AS states_with_data,
           SUM(Tot_Clms) AS tot_clms
    FROM part_d_geo
    WHERE Prscrbr_Geo_Lvl = 'State'
    GROUP BY Gnrc_Name
    ORDER BY states_with_data DESC
""", conn)

print(f"Drugs with full coverage (>=50 states): {len(state_coverage[state_coverage['states_with_data'] >= 50])}")
print(f"Drugs with near-full coverage (>=40 states): {len(state_coverage[state_coverage['states_with_data'] >= 40])}")
print(f"Drugs with limited coverage (<20 states): {len(state_coverage[state_coverage['states_with_data'] < 20])}")

state_coverage.head(20)

## Step 4: Suppression Rate

CMS suppresses values when fewer than 11 beneficiaries received a drug in a given state. The overall suppression rate informs how many gaps to expect in geographic maps for low-volume drugs.

In [ ]:
suppression = pd.read_sql_query("""
    SELECT
        COUNT(*) AS total_state_rows,
        SUM(CASE WHEN Tot_Clms IS NULL THEN 1 ELSE 0 END) AS suppressed_rows,
        ROUND(100.0 * SUM(CASE WHEN Tot_Clms IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_suppressed
    FROM part_d_geo
    WHERE Prscrbr_Geo_Lvl = 'State'
""", conn)

print(suppression)